# 12 — Text Generation — Solution

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from src.models.transformer import TransformerLM, TransformerConfig
from src.generation.sampling import greedy_decode, sample_decode

torch.manual_seed(1337)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Setup — Train Tiny Char Model

In [ ]:
corpus = """
once upon a time there was a little girl named lily. she lived in a small house near the forest.
every day she walked through the trees and listened to the birds sing. one morning she found a small
rabbit sitting by the path. the rabbit looked at her with big bright eyes. lily sat down slowly so
as not to scare it. they became the best of friends. every morning lily brought carrots from her
garden and the rabbit came to meet her by the old oak tree. they played together until the sun set.
""".strip().lower()

chars  = sorted(set(corpus))
VOCAB  = len(chars)
c2i    = {c: i for i, c in enumerate(chars)}
i2c    = {i: c for i, c in enumerate(chars)}
encode = lambda s: [c2i[c] for c in s if c in c2i]
decode = lambda ids: ''.join(i2c.get(i, '?') for i in ids)

SEQ_LEN = 48
ids  = torch.tensor(encode(corpus), dtype=torch.long)
seqs = torch.stack([ids[i:i+SEQ_LEN] for i in range(0, len(ids)-SEQ_LEN, 4)])

cfg = TransformerConfig(
    vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=3,
    d_ff=256, max_seq_len=SEQ_LEN, dropout=0.1,
    ffn_type='standard', norm_type='layernorm', pos_encoding='sinusoidal',
)
model = TransformerLM(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3, weight_decay=0.01)
loader = DataLoader(TensorDataset(seqs), batch_size=16, shuffle=True)

for epoch in range(50):
    for (batch,) in loader:
        batch = batch.to(device)
        logits = model(batch[:, :-1])
        loss = F.cross_entropy(logits.view(-1, VOCAB), batch[:, 1:].reshape(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    if (epoch+1) % 10 == 0: print(f'Epoch {epoch+1}  loss={loss.item():.4f}')

## Part A — Greedy

In [ ]:
prompt = "once upon a time"
prompt_ids = encode(prompt)

@torch.no_grad()
def greedy_generate(model, prompt_ids, max_new_tokens=80):
    model.eval()
    ids = torch.tensor([prompt_ids], device=device)
    generated = []
    for _ in range(max_new_tokens):
        logits = model(ids[:, -cfg.max_seq_len:])[:, -1, :]   # (1, V)
        next_token = logits.argmax(-1)                          # (1,)
        generated.append(next_token.item())
        ids = torch.cat([ids, next_token.unsqueeze(0)], dim=1)
    return generated

print(f'Greedy: "{decode(greedy_generate(model, prompt_ids))}"')

## Part B — Temperature

In [ ]:
@torch.no_grad()
def temperature_generate(model, prompt_ids, max_new_tokens=80, temperature=1.0):
    model.eval()
    ids = torch.tensor([prompt_ids], device=device)
    generated = []
    for _ in range(max_new_tokens):
        logits = model(ids[:, -cfg.max_seq_len:])[:, -1, :] / temperature
        probs  = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)   # (1, 1)
        generated.append(next_token.item())
        ids = torch.cat([ids, next_token], dim=1)
    return generated

for T in [0.5, 1.0, 1.5]:
    print(f'T={T}: "{decode(temperature_generate(model, prompt_ids, temperature=T))}"\n')

## Part C — Top-k Sampling

In [ ]:
def top_k_sample(logits, k, temperature=1.0):
    logits = logits / temperature
    # Keep only top-k values
    top_vals, _ = torch.topk(logits, k)
    threshold   = top_vals[..., -1, None]   # kth largest value
    logits      = logits.masked_fill(logits < threshold, float('-inf'))
    probs       = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

@torch.no_grad()
def topk_generate(model, prompt_ids, max_new_tokens=80, k=10, temperature=0.8):
    model.eval()
    ids = torch.tensor([prompt_ids], device=device)
    generated = []
    for _ in range(max_new_tokens):
        logits = model(ids[:, -cfg.max_seq_len:])[:, -1, :].squeeze(0)
        next_token = top_k_sample(logits, k=k, temperature=temperature)
        generated.append(next_token.item())
        ids = torch.cat([ids, next_token.view(1, 1)], dim=1)
    return generated

for k in [3, 10, 20]:
    print(f'top-{k}: "{decode(topk_generate(model, prompt_ids, k=k))}"\n')

## Part D — Nucleus (Top-p) Sampling

In [ ]:
def top_p_sample(logits, p, temperature=1.0):
    logits = logits / temperature
    probs  = F.softmax(logits, dim=-1)

    # Sort descending
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)

    # Remove tokens after cumulative prob exceeds p
    # Shift right so the token that pushes us over p is included
    remove = cum_probs - sorted_probs > p
    sorted_probs[remove] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum()   # renormalise

    # Sample and map back to original indices
    sampled = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_idx[sampled]

@torch.no_grad()
def topp_generate(model, prompt_ids, max_new_tokens=80, p=0.9, temperature=0.8):
    model.eval()
    ids = torch.tensor([prompt_ids], device=device)
    generated = []
    for _ in range(max_new_tokens):
        logits = model(ids[:, -cfg.max_seq_len:])[:, -1, :].squeeze(0)
        next_token = top_p_sample(logits, p=p, temperature=temperature)
        generated.append(next_token.item())
        ids = torch.cat([ids, next_token.view(1, 1)], dim=1)
    return generated

for p in [0.5, 0.9, 0.99]:
    print(f'top-p={p}: "{decode(topp_generate(model, prompt_ids, p=p))}"\n')

## Part E — Library

In [ ]:
prompt_tensor = torch.tensor([prompt_ids], device=device)

greedy_out = greedy_decode(model, prompt_tensor, max_new_tokens=60, eos_token_id=None)
print('Greedy:', decode(greedy_out[0, len(prompt_ids):].tolist()))

sampled_out = sample_decode(model, prompt_tensor, max_new_tokens=60,
                            temperature=0.8, top_k=10, top_p=0.9)
print('\nSample:', decode(sampled_out[0, len(prompt_ids):].tolist()))